# EDA — Heineken Demand Forecasting

Two things I want to get out of this:
1. How clean is the data, and what do I need to fix before modelling?
2. Are there any patterns worth knowing about — seasonality, promo effects, anything that should influence feature engineering?

## Setup

In [ ]:
%matplotlib inline
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

from utils_updated import (
    read_demand, read_promotions, extend_promotions_days,
    merge, clean_demand_per_group, aggregate_to_weekly,
    PROMOTION_DURATION_DAYS
)

np.random.seed(42)

## 1. Load the data

Two files:
- `demand.csv` — daily sales per SKU × supermarket, 2019 through 2021
- `promotions.csv` — promotion start dates (each runs for 7 days)

In [ ]:
demand = read_demand("./demand.csv")
promotions = read_promotions("./promotions.csv")

print(f"demand shape:     {demand.shape}")
print(f"promotions shape: {promotions.shape}")
demand.head()

In [ ]:
promotions.head()

## 2. Data Quality

### Basic overview

In [ ]:
print(f"Date range: {demand.index.min().date()} to {demand.index.max().date()}")
print(f"SKUs: {demand.sku.unique().tolist()}")
print(f"Supermarkets: {demand.supermarket.unique().tolist()}")
print(f"Total rows: {len(demand):,}")
print()
print(f"Missing demand values: {demand.demand.isnull().sum()} ({demand.demand.isnull().mean()*100:.1f}%)")
print()
# break down the nulls by group to see if it's concentrated anywhere
null_by_group = demand[demand.demand.isnull()].groupby(['sku', 'supermarket']).size()
print("Nulls per SKU × store:")
print(null_by_group.to_string())

In [ ]:
# sanity check: how many rows should there be vs how many there are
# 3 SKUs × 3 stores × number of days in range
n_days = (demand.index.max() - demand.index.min()).days + 1
expected = 3 * 3 * n_days
print(f"Expected rows: {expected:,}")
print(f"Actual rows:   {len(demand):,}")
print(f"Missing:       {expected - len(demand):,}")

So there are two issues: some rows are missing entirely (the dates just don't appear), and some rows exist but have NaN demand. About 11% of values need to be filled.

### Outlier check

Using z-score per group — a global z-score would be misleading since the SKUs have very different average volumes.

In [ ]:
outlier_rows = []
for (sku, sm), grp in demand.groupby(['sku', 'supermarket']):
    z = (grp.demand - grp.demand.mean()) / grp.demand.std()
    n_out = (z.abs() > 3).sum()
    outlier_rows.append({
        'SKU': sku,
        'Supermarket': sm,
        'Mean': round(grp.demand.mean(), 1),
        'Max': grp.demand.max(),
        'Outliers (|z|>3)': n_out,
        'Outlier %': round(n_out / len(grp) * 100, 1)
    })

pd.DataFrame(outlier_rows).set_index(['SKU', 'Supermarket'])

The outliers are all in the 0.5–1% range. These are almost certainly promo weeks — high demand that's real, not data errors. So I'll keep them and let the promotion flag explain them in the model.

### Imputation

For the missing values, a few options:

- Drop them — easiest but loses ~11% of data and creates gaps in the time series
- Forward fill — propagates the last known value, but can carry stale numbers too far into a gap
- **Backward fill** — pulls the next known value backwards, works well here since gaps tend to be short
- Group mean as a fallback — for anything bfill can't reach (e.g. NaNs at the tail)

Going with bfill + group mean fallback, applied per group. Applying it globally would mix up demand levels from completely different SKU×store combinations.

In [ ]:
cleaned_demand = clean_demand_per_group(demand.copy())
print(f"NaN count before: {demand.demand.isnull().sum()}")
print(f"NaN count after:  {cleaned_demand.demand.isnull().sum()}")

### Merge with promotions

Promo data only has start dates, so need to expand each to 7 days before joining.

In [ ]:
extended_promos = extend_promotions_days(promotions, PROMOTION_DURATION_DAYS)
extended_promos = extended_promos.drop("promotion_id", axis=1)

df_daily = merge(cleaned_demand, extended_promos)
print(f"Merged shape: {df_daily.shape}")
print(f"Promotion days: {df_daily.promotion.sum()} ({df_daily.promotion.mean()*100:.1f}% of all days)")
df_daily.head()

## 3. Exploration

### Aggregate to weekly

The business wants 8-week-ahead forecasts, so weekly is the right unit. Also reduces noise — daily demand is spiky, weekly averages out a lot of that.

In [ ]:
weekly = aggregate_to_weekly(df_daily)
print(f"Weekly data shape: {weekly.shape}")
print()
print("Weeks per SKU × store:")
print(weekly.groupby(['sku', 'supermarket']).size().unstack())
weekly.head()

### Demand over time

In [ ]:
from IPython.display import Image
Image('eda_fig2_timeseries.png')

In [ ]:
stats = weekly.groupby(['sku', 'supermarket'])['demand'].agg(['mean', 'std', 'min', 'max', 'count'])
stats.columns = ['Mean', 'Std', 'Min', 'Max', 'Weeks']
stats['CV'] = (stats['Std'] / stats['Mean']).round(2)  # coefficient of variation — how volatile is each series
stats.round(1)

Desperados has the highest CV (~0.29) — most volatile, hardest to forecast. Heineken Regular is the most stable (CV ~0.13).

### Seasonality and promo effect

In [ ]:
Image('eda_fig3_seasonality_promo.png')

In [ ]:
# how much does a promo actually move demand?
print("Promotion lift by SKU:")
for sku in weekly.sku.unique():
    grp = weekly[weekly.sku == sku]
    on  = grp[grp.promotion == True]['demand'].mean()
    off = grp[grp.promotion == False]['demand'].mean()
    lift = (on / off - 1) * 100
    print(f"  {sku:<22}: {off:.0f} → {on:.0f}  (+{lift:.1f}%)")

Desperados gets a +14% bump from promos, which is meaningful. The other two are lower (5–6%) but still worth including.

Monthly pattern is pretty flat — nothing like the summer spike you'd see with ice cream or sunscreen. So calendar features matter less here than the lag features will.

### Feature candidates going into modelling

Based on all of the above:
- Lag features at 8, 9, 10, 12, 13 weeks (minimum lag = 8 to avoid leakage)
- 52-week lag (same week last year)
- Rolling mean + std over 4, 8, 13 weeks (trend and volatility signal)
- Promotion flag — 14% lift on Desperados makes this important
- Month, week-of-year — mild seasonality but worth including
- SKU and store encoded as integers (so a global model can handle all 9 series)

## 4. Summary

| | |
|---|---|
| Data coverage | 3 years, 9 series (3 SKUs × 3 stores), ~156 weeks each |
| Missing values | ~11% — filled with bfill + group mean per series |
| Hardest to forecast | Desperados (CV = 0.29) |
| Promo effect | Yes, up to +14% — must be in the model |
| Seasonality | Mild — lag features likely more useful than calendar features |
| Ready for modelling? | Yes, after the cleaning and weekly aggregation above |